# Name That LLM – ModernBERT Stufe 2 (Pool → train.csv)

Startet vom Stufe-1-Modell (nur Pool trainiert) und trainiert auf der echten `train.csv` weiter. 5-Fold-CV nach Prompt für die Messung, danach ein finales Modell auf allen Daten.

In [ ]:
EPOCHS = 2
LR = 2e-05
BATCH_SIZE = 16
MAX_LEN = 384
WARMUP = 0.1
WEIGHT_DECAY = 0.01
FULL_FIT = True          # finales Modell auf allen train.csv-Texten trainieren und speichern
SEED = 42
# Zuordnung prompt_id -> Fold (identisch mit src/cv.get_folds, lokal erzeugt)
FOLD_OF_PROMPT = {"p081": 3, "p073": 1, "p120": 4, "p097": 2, "p065": 4, "p090": 4, "p036": 3, "p011": 3, "p139": 0, "p106": 3, "p003": 1, "p019": 0, "p062": 2, "p142": 2, "p108": 1, "p047": 2, "p022": 2, "p101": 3, "p046": 3, "p018": 1, "p099": 0, "p134": 0, "p051": 3, "p082": 2, "p013": 1, "p014": 0, "p023": 1, "p068": 1, "p043": 1, "p141": 3, "p131": 3, "p049": 0, "p143": 1, "p001": 2, "p008": 1, "p089": 0, "p015": 4, "p024": 0, "p083": 1, "p148": 1, "p118": 1, "p095": 4, "p102": 2, "p146": 3, "p130": 4, "p009": 0, "p091": 3, "p128": 1, "p012": 2, "p053": 1, "p137": 2, "p144": 0, "p040": 4, "p039": 0, "p140": 4, "p028": 1, "p132": 2, "p145": 4, "p086": 3, "p116": 3, "p056": 3, "p070": 4, "p006": 3, "p093": 1, "p117": 2, "p071": 3, "p126": 3, "p025": 4, "p035": 4, "p136": 3, "p030": 4, "p109": 0, "p124": 0, "p059": 0, "p077": 2, "p100": 4, "p138": 1, "p063": 1, "p114": 0, "p113": 1, "p007": 2, "p107": 2, "p052": 2, "p087": 2, "p042": 2, "p060": 4, "p002": 4, "p103": 1, "p045": 4, "p104": 0, "p038": 1, "p123": 1, "p069": 0, "p004": 0, "p074": 0, "p017": 2, "p110": 4, "p133": 1, "p033": 1, "p037": 2, "p129": 0, "p067": 2, "p054": 0, "p079": 0, "p076": 3, "p135": 4, "p021": 3, "p115": 4, "p147": 2, "p119": 0, "p061": 3, "p094": 0, "p127": 2, "p088": 1, "p044": 0, "p072": 2, "p005": 4, "p016": 3, "p000": 3, "p057": 2, "p096": 3, "p080": 4, "p125": 4, "p048": 1, "p032": 2, "p026": 3, "p075": 4, "p105": 4, "p020": 4, "p055": 4, "p027": 2, "p111": 3, "p041": 3, "p092": 2, "p010": 4, "p084": 0, "p064": 0, "p034": 0, "p085": 4, "p098": 1, "p066": 3, "p112": 2, "p121": 3, "p050": 4, "p078": 1, "p031": 3, "p029": 0, "p149": 0, "p058": 1, "p122": 2}

In [ ]:
import glob, math, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import balanced_accuracy_score
from torch.utils.data import DataLoader
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, get_linear_schedule_with_warmup)

assert torch.cuda.is_available(), "GPU aktivieren"
device = torch.device("cuda")

def find(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"{pattern} nicht gefunden"
    return Path(hits[0])

TRAIN_CSV = find("name-that-llm/train.csv")
STAGE1 = find("model/config.json").parent
STAGE1_PROBA = find("train_proba.parquet")
OUT = Path("/kaggle/working")
print(TRAIN_CSV, STAGE1)

real = pd.read_csv(TRAIN_CSV)
folds = real["prompt_id"].map(FOLD_OF_PROMPT).values
assert not np.isnan(folds.astype(float)).any()
tokenizer = AutoTokenizer.from_pretrained(STAGE1)
probe = AutoModelForSequenceClassification.from_pretrained(STAGE1)
classes = np.array([probe.config.id2label[i] for i in range(probe.config.num_labels)])
del probe
label_id = {c: i for i, c in enumerate(classes)}
y = real["model"].map(label_id).values
FAMILIES = ("claude", "gpt", "gemini", "glm", "kimi", "qwen")
family = lambda m: next(f for f in FAMILIES if m.startswith(f))
print(len(real), "Texte | Folds:", np.bincount(folds))

In [ ]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels=None):
        self.enc = tokenizer(list(texts), truncation=True, max_length=MAX_LEN)
        self.labels = labels
    def __len__(self):
        return len(self.enc["input_ids"])
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        if self.labels is not None:
            item["labels"] = int(self.labels[i])
        return item

collate = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)
ds_all = TextDataset(real["text"], y)

@torch.no_grad()
def predict(model, idx):
    model.eval()
    dl = DataLoader(torch.utils.data.Subset(ds_all, idx), batch_size=64, collate_fn=collate)
    out = []
    for b in dl:
        b = {k: v.to(device) for k, v in b.items() if k != "labels"}
        with torch.autocast("cuda", dtype=torch.float16):
            out.append(torch.softmax(model(**b).logits.float(), -1).cpu().numpy())
    model.train()
    return np.concatenate(out)

def train(tr_idx, va_idx=None):
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(STAGE1).float().to(device)
    no_decay = ("bias", "LayerNorm.weight", "norm.weight")
    params = [
        {"params": [p for n, p in model.named_parameters() if not n.endswith(no_decay)], "weight_decay": WEIGHT_DECAY},
        {"params": [p for n, p in model.named_parameters() if n.endswith(no_decay)], "weight_decay": 0.0},
    ]
    optim = torch.optim.AdamW(params, lr=LR)
    dl = DataLoader(torch.utils.data.Subset(ds_all, tr_idx), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
    steps = len(dl) * EPOCHS
    sched = get_linear_schedule_with_warmup(optim, int(WARMUP * steps), steps)
    scaler = torch.amp.GradScaler("cuda")
    per_epoch = []
    if va_idx is not None:
        per_epoch.append(predict(model, va_idx))          # Epoche 0 = reines Stufe-1-Modell
    model.train()
    for epoch in range(EPOCHS):
        for b in dl:
            b = {k: v.to(device) for k, v in b.items()}
            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(**b).loss
            optim.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optim)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optim); scaler.update(); sched.step()
        if va_idx is not None:
            per_epoch.append(predict(model, va_idx))
    return model, per_epoch

## 5-Fold-CV

In [ ]:
t0 = time.time()
oof = np.zeros((EPOCHS + 1, len(real), len(classes)), dtype=np.float32)
for k in range(5):
    tr_idx, va_idx = np.where(folds != k)[0], np.where(folds == k)[0]
    model, per_epoch = train(tr_idx, va_idx)
    for e, p in enumerate(per_epoch):
        oof[e, va_idx] = p
    accs = [balanced_accuracy_score(y[va_idx], p.argmax(1)) for p in per_epoch]
    print(f"Fold {k}: " + " | ".join(f"Ep{e} {a:.4f}" for e, a in enumerate(accs)) + f" | {(time.time()-t0)/60:.1f} min", flush=True)
    del model; torch.cuda.empty_cache()

rows = []
for e in range(EPOCHS + 1):
    pred = classes[oof[e].argmax(1)]
    rows.append({"epoch": e, "balanced_accuracy": balanced_accuracy_score(real["model"], pred),
                 "family_accuracy": np.mean([family(a) == family(b) for a, b in zip(real["model"], pred)])})
    out = pd.DataFrame(oof[e], columns=classes); out.insert(0, "id", real["id"].values)
    out.to_parquet(OUT / f"oof_stage2_ep{e}.parquet", index=False)
res = pd.DataFrame(rows); res.to_csv(OUT / "cv_stage2.csv", index=False)
print("OOF gesamt (Epoche 0 = nur Stufe 1):"); res

## Finales Modell auf allen train.csv-Texten

In [ ]:
if FULL_FIT:
    model, _ = train(np.arange(len(real)))
    model.save_pretrained(OUT / "model_stage2")
    tokenizer.save_pretrained(OUT / "model_stage2")
    print("gespeichert:", OUT / "model_stage2", f"| gesamt {(time.time()-t0)/60:.1f} min")